# Real-Time Face Recognition: Step by Step

This notebook teaches how the completed project works. It combines computer-vision ideas with the actual repository code. It uses only explicitly enrolled, consenting participants, keeps processing local, and does not identify strangers from outside databases.

```text
Webcam / image
      ↓
Face detection
      ↓
Face crop / location
      ↓
128-D face embedding
      ↓
Compare with enrolled embeddings
      ↓
Euclidean distance
      ↓
Distance threshold
      ↓
Known person / Unknown
```

The question is deliberately limited: **Is the visible face sufficiently similar to someone explicitly enrolled in this local dataset?**

## 1. Notebook setup and libraries

OpenCV (`cv2`) reads images and webcams and draws overlays. NumPy represents images and embeddings as arrays. `face_recognition` supplies dlib's HOG face detector and 128-dimensional encoder. Pandas organizes experiment CSV files, while Matplotlib plots real measurements.

Run the notebook from either the repository root or the `notebooks/` folder. Missing optional packages are reported clearly. Install `requirements.txt` before the full walkthrough.

In [ ]:
from pathlib import Path
import sys
import time
import numpy as np

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

missing = []
try:
    import cv2
except ImportError:
    cv2 = None
    missing.append('opencv-python')
try:
    import face_recognition
except ImportError:
    face_recognition = None
    missing.append('face-recognition')
try:
    import pandas as pd
except ImportError:
    pd = None
    missing.append('pandas')
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    missing.append('matplotlib')

print('Project root:', ROOT)
print('Missing packages:' if missing else 'All notebook packages are available.', ', '.join(missing))

In [ ]:
from face_database import (
    DEFAULT_DATABASE_PATH, DEFAULT_THRESHOLD, UNKNOWN_LABEL,
    euclidean_distance, load_database, match_embedding,
)
from face_detector import detect_faces, face_size, largest_face
from face_encoder import IMAGE_SUFFIXES, get_face_embedding

print('Actual project threshold:', DEFAULT_THRESHOLD)
print('Metric: Euclidean distance (lower means more similar)')
print('Database path:', ROOT / DEFAULT_DATABASE_PATH)

## Today's local demo workflow

This section uses the permitted images under `data/enrolled/Obama` and `data/enrolled/Elon`. These are user-defined labels for consenting local participants. Each currently has ten enrollment images. Enrollment images teach the pipeline here but are not reused to claim held-out accuracy.

### A. Load one reference image

Find the first supported image under `Obama`, load it with OpenCV, convert BGR to RGB, and display only that locally supplied image.

In [ ]:
person_a_paths = sorted(p for p in (ROOT / 'data/enrolled/Obama').rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES)
reference_a_rgb = None
if cv2 is None or plt is None:
    print('Install OpenCV and Matplotlib to run this cell.')
elif not person_a_paths:
    print('DATA COLLECTION REQUIRED: add 5-10 permitted images to data/enrolled/Obama/.')
else:
    reference_a_bgr = cv2.imread(str(person_a_paths[0]))
    if reference_a_bgr is None:
        print('The selected image could not be opened:', person_a_paths[0])
    else:
        reference_a_rgb = cv2.cvtColor(reference_a_bgr, cv2.COLOR_BGR2RGB)
        print('Loaded:', person_a_paths[0].relative_to(ROOT), 'shape:', reference_a_rgb.shape)
        plt.imshow(reference_a_rgb); plt.axis('off'); plt.show()

### B. Detect the face

`detect_faces()` returns `(top, right, bottom, left)` boxes. Enrollment expects exactly one face per reference image.

In [ ]:
reference_a_boxes = []
if reference_a_rgb is None or face_recognition is None:
    print('DATA COLLECTION REQUIRED or face-recognition is not installed.')
else:
    reference_a_boxes = detect_faces(reference_a_rgb)
    print('Faces detected:', len(reference_a_boxes), reference_a_boxes)
    boxed = reference_a_rgb.copy()
    for top, right, bottom, left in reference_a_boxes:
        cv2.rectangle(boxed, (left, top), (right, bottom), (0, 255, 0), 2)
    plt.imshow(boxed); plt.axis('off'); plt.show()

### C. Extract an embedding

The project encodes one detected face into a 128-value vector. We print its shape, not hundreds of sensitive derived values.

In [ ]:
embedding_a1 = None
if len(reference_a_boxes) == 1:
    embedding_a1 = get_face_embedding(reference_a_rgb, reference_a_boxes[0])
    print('Embedding shape:', None if embedding_a1 is None else embedding_a1.shape)
else:
    print('Exactly one detected reference face is required.')

### D–E. Load a second image of the same local identity and compare

A second, distinct `Obama` image tests within-label variation. Smaller Euclidean distance means closer embeddings.

In [ ]:
embedding_a2 = None
same_identity_distance = None
if cv2 is None or len(person_a_paths) < 2:
    print('DATA COLLECTION REQUIRED: add at least two permitted Obama images.')
else:
    second_bgr = cv2.imread(str(person_a_paths[1]))
    second_rgb = cv2.cvtColor(second_bgr, cv2.COLOR_BGR2RGB) if second_bgr is not None else None
    second_boxes = detect_faces(second_rgb) if second_rgb is not None and face_recognition is not None else []
    if len(second_boxes) == 1:
        embedding_a2 = get_face_embedding(second_rgb, second_boxes[0])
    if embedding_a1 is not None and embedding_a2 is not None:
        same_identity_distance = euclidean_distance(embedding_a1, embedding_a2)
        print('Obama image 1 vs image 2 distance:', round(same_identity_distance, 4))
    else:
        print('Both images must contain exactly one encodable face.')

### F. Compare different local identities

The cell compares the first permitted `Obama` reference with the first permitted `Elon` reference. A larger cross-label distance is common, but not guaranteed for every image; distances are statistical model outputs.

In [ ]:
other_paths = sorted(p for p in (ROOT / 'data/enrolled/Elon').rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES)
different_identity_distance = None
if cv2 is None or not other_paths:
    print('DATA COLLECTION REQUIRED: add a permitted image to data/enrolled/Elon/.')
else:
    other_bgr = cv2.imread(str(other_paths[0]))
    other_rgb = cv2.cvtColor(other_bgr, cv2.COLOR_BGR2RGB) if other_bgr is not None else None
    other_boxes = detect_faces(other_rgb) if other_rgb is not None and face_recognition is not None else []
    other_embedding = get_face_embedding(other_rgb, other_boxes[0]) if len(other_boxes) == 1 else None
    if embedding_a1 is not None and other_embedding is not None:
        different_identity_distance = euclidean_distance(embedding_a1, other_embedding)
        print('Obama vs Elon distance:', round(different_identity_distance, 4))
        if same_identity_distance is not None:
            print('Same-label distance is smaller in this observation:', same_identity_distance < different_identity_distance)
    else:
        print('Both local labels need one encodable face.')

### G. Apply the threshold

For this Euclidean metric, `distance <= 0.60 → known`; `distance > 0.60 → Unknown`. This is not a probability. Tune the threshold only with real known and unknown validation observations.

In [ ]:
for description, measured_distance in [('same local identity', same_identity_distance), ('different local identity', different_identity_distance)]:
    if measured_distance is None:
        print(description, '— DATA COLLECTION REQUIRED')
    else:
        decision = 'Known candidate' if measured_distance <= DEFAULT_THRESHOLD else 'Unknown'
        print(f'{description}: distance={measured_distance:.4f}, threshold={DEFAULT_THRESHOLD:.2f}, decision={decision}')

### H. Load the enrollment database

First run `python face_database.py --build`. Multiple reference embeddings are retained for each local label.

In [ ]:
demo_database_path = ROOT / DEFAULT_DATABASE_PATH
if demo_database_path.exists():
    demo_database = load_database(demo_database_path)
    print('Stored local identities:', {name: len(vectors) for name, vectors in demo_database.items()})
else:
    demo_database = {}
    print('DATA COLLECTION REQUIRED: run python face_database.py --build')

### I. Recognize one saved image

Place a different permitted test image in `examples/known/` or `examples/unknown/`. This cell applies the same `match_embedding()` logic as the script. The terminal equivalent is `python recognize.py --image examples/known/test.jpg --debug`.

In [ ]:
demo_test_paths = sorted(p for folder in [ROOT / 'examples/known', ROOT / 'examples/unknown'] for p in folder.rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES)
if cv2 is None or not demo_test_paths or not demo_database:
    print('DATA COLLECTION REQUIRED: add a permitted test image and build the database.')
else:
    test_bgr = cv2.imread(str(demo_test_paths[0]))
    test_rgb = cv2.cvtColor(test_bgr, cv2.COLOR_BGR2RGB) if test_bgr is not None else None
    test_boxes = detect_faces(test_rgb) if test_rgb is not None and face_recognition is not None else []
    if not test_boxes:
        print('No face detected in:', demo_test_paths[0])
    else:
        for index, box in enumerate(test_boxes, 1):
            query_embedding = get_face_embedding(test_rgb, box)
            if query_embedding is not None:
                label, distance, closest = match_embedding(query_embedding, demo_database, DEFAULT_THRESHOLD)
                print(f'Face {index}: prediction={label}, closest={closest}, distance={distance:.4f}, threshold={DEFAULT_THRESHOLD:.2f}')

### J. Webcam mode

Jupyter webcam windows are unreliable, so run `python recognize.py` in a terminal. The script reads frames, resizes them, detects every face, extracts embeddings, compares all references, applies the threshold independently, draws labels/distances, and displays measured FPS. Use `python recognize.py --debug` for terminal details and press `Q` to quit. A face photograph displayed on a phone is useful for demonstrating this pipeline, but it is not proof of a live person and there is no liveness detection.

## 2. How the project files work together

```text
enroll.py → data/enrolled/<name>/frame_*.jpg
                         ↓
face_database.py → face_encoder.py → embeddings/face_embeddings.npz
                                      ↓
recognize.py → face_detector.py → face_encoder.py → face_database.py
                                      ↓
                              Known / Unknown

evaluate.py → outputs/metrics/ + outputs/figures/
benchmark.py → outputs/metrics/performance_metrics.csv
```

| File or folder | Purpose | Called by / output |
|---|---|---|
| `enroll.py` | Opens the webcam, checks one face, size, and blur, then saves selected stills | Human runs it; writes `data/enrolled/` |
| `face_detector.py` | Wraps HOG detection and box helpers | Encoder, live recognition, evaluation, benchmark |
| `face_encoder.py` | Produces one embedding or embeds a folder | Database builder and recognition paths |
| `face_database.py` | Builds, saves, loads, and matches references | Writes/reads the local NPZ; used by recognition/evaluation |
| `recognize.py` | Runs the multi-face webcam loop and draws labels/FPS | Human runs it; reads embeddings |
| `evaluate.py` | Evaluates labeled consented images | Writes real CSV/JSON metrics and supported plots |
| `benchmark.py` | Times detection, encoding, and matching on stored images | Writes real performance measurements |
| `utils.py` | Path creation, safe names, color conversion, blur, and box clamping | Small helpers used by scripts |
| `tests/` | Synthetic distance, matching, and NPZ round-trip tests | `pytest` |
| `data/` | Private enrollment and test images | Ignored by Git except placeholders |
| `embeddings/` | Sensitive derived biometric references | Ignored by Git |
| `outputs/` | Metrics, figures, and optional examples | Ignored by default for privacy review |
| `README.md` | Setup, commands, architecture, and limitations | Main project entry point |
| `LEARNING_GUIDE.md` | Beginner concept reference | Companion to this notebook |
| `EXPERIMENTS.md` | Controlled collection protocol | Guides real experiments |

## 3. What is a digital image?

A color image is a grid of pixels. Width counts columns, height counts rows, and each pixel has color-channel values. OpenCV stores color as **BGR**; many plotting/model tools expect **RGB**. A grayscale image has one brightness value per pixel.

A webcam frame shaped `(480, 640, 3)` therefore has height 480, width 640, and three color channels.

In [ ]:
tiny_bgr = np.zeros((3, 4, 3), dtype=np.uint8)
tiny_bgr[1, 2] = [255, 0, 0]  # blue in OpenCV's BGR order
print('shape:', tiny_bgr.shape)
print('height:', tiny_bgr.shape[0], 'width:', tiny_bgr.shape[1], 'channels:', tiny_bgr.shape[2])
print('one BGR pixel:', tiny_bgr[1, 2])

## 4. Webcam frame capture

`cv2.VideoCapture(0)` requests camera index 0. `ret, frame = cap.read()` returns a Boolean `ret` saying whether capture succeeded and a NumPy image `frame`. Cameras produce a stream of frames, often tens per second. Always release the device.

Interactive webcam windows can behave poorly inside Jupyter, so the cell below is intentionally a non-executed learning example. Use `python recognize.py` for live testing.

```python
cap = cv2.VideoCapture(0)
ret, frame = cap.read()
if ret:
    print(frame.shape)
cap.release()
```

## 5. Face detection

Detection asks **where is the face?** Recognition asks **whose enrolled face is it?** `detect_faces(rgb_image)` returns every face as `(top, right, bottom, left)`. `largest_face()` is useful when a single-person image needs one box, while live recognition loops over all boxes. `face_size()` returns width and height.

The next cell visualizes boxes only when a real consented test image and the backend exist.

In [ ]:
test_images = sorted(
    p for p in (ROOT / 'data' / 'test').rglob('*')
    if p.suffix.lower() in IMAGE_SUFFIXES
)
if cv2 is None or face_recognition is None or plt is None:
    print('Install OpenCV, face_recognition, and Matplotlib to run detection visualization.')
elif not test_images:
    print('DATA COLLECTION REQUIRED: no consented test image is available.')
else:
    sample_path = test_images[0]
    sample_bgr = cv2.imread(str(sample_path))
    sample_rgb = cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2RGB)
    sample_boxes = detect_faces(sample_rgb)
    shown = sample_rgb.copy()
    for top, right, bottom, left in sample_boxes:
        cv2.rectangle(shown, (left, top), (right, bottom), (0, 255, 0), 2)
    print('Image:', sample_path.relative_to(ROOT), 'boxes:', sample_boxes)
    plt.figure(figsize=(7, 5)); plt.imshow(shown); plt.axis('off'); plt.show()

## 6. Face cropping

An image array is indexed as rows first, then columns. For `(top, right, bottom, left)`, a crop is `image[top:bottom, left:right]`. The crop keeps the facial region instead of unrelated background. In this project the encoder receives the full RGB image plus the known face box; the library internally focuses encoding on that location, so manual cropping is mainly useful for visualization.

In [ ]:
if 'sample_boxes' in globals() and sample_boxes and plt is not None:
    top, right, bottom, left = sample_boxes[0]
    face_crop = sample_rgb[top:bottom, left:right]
    print('Full image:', sample_rgb.shape, 'crop:', face_crop.shape)
    plt.imshow(face_crop); plt.axis('off'); plt.show()
else:
    print('A detected, consented test face and Matplotlib are required to display a crop.')

## 7. Face embeddings

A face embedding is a numerical description of facial appearance:

```text
face image → learned encoder → [0.12, -0.43, 0.81, ...]
```

The dlib model used here returns a one-dimensional vector with 128 values. Individual coordinates do **not** directly mean eye color, nose size, or another human-named feature. The learned values work collectively. Images of the same person are generally relatively close and different people generally farther apart, but this is statistical similarity—not certainty.

In [ ]:
if 'sample_boxes' in globals() and sample_boxes:
    sample_embedding = get_face_embedding(sample_rgb, sample_boxes[0])
    if sample_embedding is None:
        print('The detected face could not be encoded.')
    else:
        print('Actual embedding shape:', sample_embedding.shape)
        print('First five values:', sample_embedding[:5])
else:
    print('DATA COLLECTION REQUIRED: an actual face is needed to verify an embedding shape.')

## 8. Comparing two face embeddings

The project uses Euclidean distance: `np.linalg.norm(a - b)`. A smaller distance means the vectors are closer and therefore more similar under the model. A larger distance means less similarity. Cosine similarity is an alternative, but it is not the metric used here and its thresholds would not be interchangeable.

In [ ]:
a = np.array([1.0, 0.0])
b = np.array([0.95, 0.05])
c = np.array([0.0, 1.0])

ab = euclidean_distance(a, b)
ac = euclidean_distance(a, c)
print(f'a vs b: {ab:.4f}')
print(f'a vs c: {ac:.4f}')
print('a and b are closer:', ab < ac)

## 9. Enrolling a person

Run `python enroll.py --name Alice` in a terminal. The camera opens, detection draws a preview box, and `SPACE` asks to save one still. The script rejects zero/multiple faces, small faces, and heavily blurred frames. It stores clean frames—not preview annotations—under the person's folder. `Q` quits.

```text
camera → detect exactly one face → quality checks → selected stills
       → data/enrolled/Alice/frame_01.jpg ...
```

Several images capture modest changes in expression, frontal/slight-left/slight-right pose, and natural appearance. They must remain good-quality references.

```text
data/enrolled/
├── Alice/
│   ├── frame_01.jpg
│   ├── frame_02.jpg
│   └── ...
└── Bob/
    └── ...
```

These private images are excluded by `.gitignore`. `Unknown` is reserved and cannot be enrolled.

## 10. Building the local face database

Run `python face_database.py --build`. `build_database()` visits every identity folder. `get_embeddings_from_folder()` loads supported images, requires exactly one detected face, and calls `get_face_embedding()`. `save_database()` writes parallel `names` and `embeddings` arrays to `embeddings/face_embeddings.npz`.

In Python, the loaded representation is:

```python
{
    'Alice': [embedding_1, embedding_2, ...],
    'Bob': [embedding_1, ...],
}
```

The project retains multiple references instead of averaging them. A query can match the closest valid view. Embeddings are still sensitive biometric representations, so the entire folder is ignored by Git and must remain protected.

In [ ]:
database_path = ROOT / DEFAULT_DATABASE_PATH
if database_path.exists():
    database = load_database(database_path)
    print({name: len(vectors) for name, vectors in database.items()})
else:
    database = {}
    print('DATA COLLECTION REQUIRED: enroll a person, then run face_database.py --build.')

## 11. Real-time recognition loop

`recognize.py` loads the database once and repeatedly reads frames. `recognize_frame()` resizes a frame, converts BGR to RGB, detects every face, encodes each location, calls `match_embedding()`, and scales boxes back to display size. It returns recognition results plus the detected-face count. The main loop reuses the latest result between processed frames and measures real timing.

```python
while True:
    ret, frame = camera.read()
    if frame_number % process_every == 0:
        results, face_count = recognize_frame(frame, database, threshold, scale)
    for box, label, distance, closest in results:
        # draw the result
        ...
```

For reliable interactive camera behavior, run `python recognize.py` rather than opening a webcam window inside this notebook.

## 12. Why `Unknown` needs a threshold

Suppose the database contains only Alice and Bob, but a different consenting test participant appears. One of Alice or Bob must still be mathematically closest. Closest does not automatically mean correct.

`match_embedding()` first finds the smallest distance across every stored reference, then applies the threshold:

```python
if best_distance <= threshold:
    label = best_name
else:
    label = 'Unknown'
```

The default maximum distance is **0.60**. A smaller threshold is stricter and can cause more false rejections. A larger threshold is more permissive and can cause more false acceptances. The closest identity is retained for analysis even when the final decision is `Unknown`.

In [ ]:
toy_database = {
    'Alice': [np.array([1.0, 0.0])],
    'Bob': [np.array([0.0, 1.0])],
}
query = np.array([-1.0, 0.0])
for threshold in [0.3, 1.5, 2.1]:
    label, distance, closest = match_embedding(query, toy_database, threshold)
    print(f'threshold={threshold:.1f}: closest={closest}, distance={distance:.3f}, decision={label}')
print('Toy dimensions illustrate logic only; 0.60 belongs to real 128-D model embeddings.')

### False acceptance, false rejection, and score wording

- **False acceptance:** an unknown person appears and the system says `Alice`.
- **False rejection:** Alice appears and the system says `Unknown`.

Changing the threshold trades one risk against the other. The overlay reports **embedding distance**. It is not a softmax score, confidence percentage, or calibrated probability, so the project never claims “98% probability Alice.”

## 13. Drawing boxes and labels

OpenCV uses `(x, y)` display coordinates. For this project's box, the rectangle corners are `(left, top)` and `(right, bottom)`. `cv2.rectangle()` draws the border and `cv2.putText()` writes identity, distance, FPS, and latency. Known boxes are green and `Unknown` boxes red. The real-image detection cell above demonstrates this only when a consented image is present.

## 14. FPS, latency, and optimization

FPS is displayed frames divided by elapsed time. `time.perf_counter()` is a high-resolution timer. Recognition latency is the duration of one detection/encoding/matching operation. For example, 45 ms is 0.045 seconds. Low latency improves responsiveness, while high FPS makes display smoother; neither guarantees accurate recognition.

`--scale 0.5` reduces image width and height before recognition, greatly reducing pixels but shrinking distant faces. `--process-every 2` recognizes every second camera frame and reuses the latest boxes between updates. These are speed/accuracy/responsiveness trade-offs, not free improvements.

In [ ]:
started = time.perf_counter()
_ = np.linalg.norm(np.ones(128) - np.zeros(128))
elapsed_ms = (time.perf_counter() - started) * 1000
print(f'Example timer measurement: {elapsed_ms:.6f} ms')
print('This measures this NumPy operation only, not webcam recognition performance.')

## 15. Loading real evaluation results

`evaluate.py` expects consented images under identity folders or a manifest containing `image`, `expected_identity`, and `condition`. It records prediction, closest identity, distance, threshold, latency, face size, detected-face count, and correctness. This notebook never creates substitute results.

In [ ]:
results_path = ROOT / 'outputs' / 'metrics' / 'evaluation_results.csv'
if pd is None:
    results = None
    print('Install pandas to analyze results.')
elif results_path.exists():
    results = pd.read_csv(results_path)
    print(f'Loaded {len(results)} actual observations.')
    display(results.head())
else:
    results = pd.DataFrame()
    print('DATA COLLECTION REQUIRED: run evaluate.py after collecting labeled, consented images.')

## 16. Experiment 1 — Lighting

Collect separate `normal`, `dim`, and `bright` observations while keeping participant, distance, and pose as stable as practical. Record expected identity, prediction, distance, and latency. Lighting can change contrast, shadows, sensor noise, detection, and the embedding.

In [ ]:
def plot_condition(group, order, title):
    if results is None or results.empty:
        print('DATA COLLECTION REQUIRED')
        return
    subset = results[results['condition'].str.lower().isin(order)].copy()
    if subset.empty:
        print(f'DATA COLLECTION REQUIRED: no {group.lower()} labels found.')
        return
    subset['condition'] = subset['condition'].str.lower()
    summary = subset.groupby('condition')['correct'].agg(['mean', 'count']).reindex(order).dropna()
    display(summary)
    if plt is not None:
        summary['mean'].plot.bar(ylim=(0, 1), title=title, rot=0)
        plt.ylabel('Recognition accuracy'); plt.show()

plot_condition('Lighting', ['normal', 'dim', 'bright'], 'Lighting observations')

## 17. Experiment 2 — Distance

Use `near`, `medium`, and `far` unless distance was physically measured. Greater distance gives the face fewer pixels, which can reduce detection and embedding detail. Compare correctness and distance together with `face_width` and `face_height`.

In [ ]:
plot_condition('Distance', ['near', 'medium', 'far'], 'Distance observations')
if results is not None and not results.empty:
    distance_rows = results[results['condition'].str.lower().isin(['near', 'medium', 'far'])]
    if not distance_rows.empty:
        display(distance_rows.groupby('condition')[['distance', 'face_width', 'face_height']].mean())

## 18. Experiment 3 — Pose

Collect `frontal`, moderate `left`, `right`, `up`, and `down` observations. Pose changes visible facial structure and can affect HOG detection, the model's implicit alignment, and embedding quality. Avoid extreme turns and do not infer a cause from one failure.

In [ ]:
plot_condition('Pose', ['frontal', 'left', 'right', 'up', 'down'], 'Pose observations')

## 19. Experiment 4 — Unknown person

Ask a consenting person who is **not enrolled** to provide test images under `data/test/Unknown/`. The desired decision is `Unknown`. This is more meaningful than testing only enrolled identities because it checks whether the threshold prevents forced matches. Do not save anyone's test image without explicit consent.

In [ ]:
if results is None or results.empty:
    print('DATA COLLECTION REQUIRED')
else:
    unknown_rows = results[results['expected_identity'].str.lower().eq('unknown')]
    if unknown_rows.empty:
        print('DATA COLLECTION REQUIRED: no unknown-person observations found.')
    else:
        display(unknown_rows[['image', 'closest_identity', 'distance', 'threshold', 'predicted_identity', 'correct']])
        print('Unknown rejection rate:', unknown_rows['predicted_identity'].str.lower().eq('unknown').mean())

## 20. Experiment 5 — Threshold tuning

Reuse held-out distances rather than re-running the encoder. With the current two enrolled labels, measure accepted correct matches, rejected known matches, and incorrect identity assignments. Only add Unknown rejection and false-acceptance rates after collecting a genuine Unknown test set.

In [ ]:
if results is None or results.empty:
    print('DATA COLLECTION REQUIRED: separate held-out images are needed.')
else:
    usable = results.dropna(subset=['distance']).copy()
    is_unknown = usable['expected_identity'].str.lower().eq('unknown')
    known = ~is_unknown
    if usable.empty or not known.any():
        print('DATA COLLECTION REQUIRED: held-out known-participant distances are needed.')
    else:
        rows = []
        for threshold in np.linspace(0.35, 0.80, 19):
            accepted = usable['distance'] <= threshold
            correct_identity = usable['closest_identity'].str.lower() == usable['expected_identity'].str.lower()
            rows.append({
                'threshold': threshold,
                'known_recognition_rate': (accepted & correct_identity)[known].mean(),
                'known_false_rejection_rate': (~accepted)[known].mean(),
                'incorrect_assignment_rate': (accepted & ~correct_identity)[known].mean(),
            })
        threshold_table = pd.DataFrame(rows).set_index('threshold')
        display(threshold_table)
        if plt is not None:
            threshold_table.plot(marker='o', ylim=(0, 1.05), title='Known-participant threshold analysis')
            plt.ylabel('Rate'); plt.show()
        if not is_unknown.any():
            print('Unknown rejection is not reported because no Unknown test set exists.')

## 21. Experiment 6 — Runtime performance

`benchmark.py` times detection, encoding, and matching on stored images with `time.perf_counter()`. Its CSV reports image count, detected faces, mean latency, median latency, and approximate FPS. Approximate FPS is `1000 / average_latency_ms`; it is not the webcam display FPS.

In [ ]:
performance_path = ROOT / 'outputs' / 'metrics' / 'performance_metrics.csv'
if pd is None:
    print('Install pandas to load benchmark results.')
elif performance_path.exists():
    performance = pd.read_csv(performance_path)
    display(performance)
else:
    print('DATA COLLECTION REQUIRED: run benchmark.py on permitted stored images; the current CSV measures enrollment-image runtime only.')

## 22. Failure cases

A failed row should be inspected, not automatically explained. Ask: Was the face small? Was lighting poor? Was pose difficult? Did detection fail? Was the image blurred? Was the threshold too strict? A coincidence does not establish a cause. The cell shows recorded fields and displays consented local images only if they still exist.

In [ ]:
if results is None or results.empty:
    print('DATA COLLECTION REQUIRED')
else:
    failures = results[~results['correct']].copy()
    if failures.empty:
        print('No failures are present in the collected results.')
    else:
        display(failures[['image', 'expected_identity', 'predicted_identity', 'distance', 'condition', 'face_width', 'faces_detected']])
        if cv2 is not None and plt is not None:
            for _, row in failures.head(3).iterrows():
                path = Path(row['image'])
                if not path.is_absolute():
                    path = ROOT / path
                image = cv2.imread(str(path))
                if image is not None:
                    plt.figure(figsize=(4, 3))
                    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis('off')
                    plt.title(f"Expected: {row['expected_identity']} | Predicted: {row['predicted_identity']}\nd={row['distance']} | {row['condition']}")
                    plt.show()

## 23. Confusion matrix

A confusion matrix has actual labels on rows and predicted labels on columns. The diagonal contains correct decisions; off-diagonal cells contain errors. `Unknown` appears when it is part of evaluation. `evaluate.py` creates the figure only for a minimally useful multi-label sample.

In [ ]:
confusion_path = ROOT / 'outputs' / 'figures' / 'confusion_matrix.png'
if confusion_path.exists() and plt is not None:
    confusion_image = plt.imread(confusion_path)
    plt.figure(figsize=(7, 6)); plt.imshow(confusion_image); plt.axis('off'); plt.show()
else:
    print('DATA COLLECTION REQUIRED: no generated confusion matrix is available.')

## 24. Function input/output guide

| Function | Input | Output |
|---|---|---|
| `detect_faces()` | RGB NumPy image, optional detector model | List of `(top, right, bottom, left)` boxes |
| `largest_face()` | Face-location sequence | Largest box or `None` |
| `face_size()` | One face box | `(width, height)` |
| `get_face_embedding()` | RGB image, optional location, jitter count | 128-D array or `None` |
| `get_embeddings_from_folder()` | Folder path | Embedding list and used-path list |
| `euclidean_distance()` | Two equal-shaped vectors | Float distance |
| `match_embedding()` | Query, `{name: [references]}`, threshold | `(final_label, best_distance, closest_name)` |
| `build_database()` | Enrollment directory | `{identity: [embeddings]}` |
| `save_database()` | Database dictionary and NPZ path | Writes local NPZ |
| `load_database()` | NPZ path | Database dictionary |
| `recognize_frame()` | BGR frame, database, threshold, scale | Results list and detected-face count |
| `collect_samples()` | Test directory and optional manifest | Labeled sample records |
| `evaluate_image()` | Sample record, database, threshold | One metrics record |
| `calculate_metrics()` | Results DataFrame | Aggregate metrics dictionary |
| `blur_score()` | Image | Laplacian variance float |
| `safe_identity_name()` | User-entered identity | Safe local folder name |

## 25. Code walkthrough by file

### `face_detector.py`
`_backend()` provides a clear installation error. `detect_faces()` wraps HOG detection. `largest_face()` supports one-subject evaluation and `face_size()` measures resolution.

### `face_encoder.py`
`get_face_embedding()` encodes a supplied location or detects the largest face. `image_files()` filters extensions. `get_embeddings_from_folder()` skips images that do not contain exactly one face.

### `face_database.py`
`euclidean_distance()` validates shapes. `match_embedding()` searches every reference and applies the threshold. `build_database()`, `save_database()`, and `load_database()` manage the local NPZ. `main()` implements `--build` and `--list`.

### `enroll.py`
`parse_args()` defines camera, quality, and sample controls. `main()` sanitizes the name, opens the camera, previews detection, accepts `SPACE`, checks face size/blur, and saves clean stills.

### `recognize.py`
`recognize_frame()` performs resized multi-face inference. `main()` validates settings, loads references, controls processing cadence, draws results, and measures FPS/latency.

### `evaluate.py`
`collect_samples()` reads folders or a manifest. `evaluate_image()` records an actual decision. `calculate_metrics()` summarizes it. `generate_figures()` creates only plots supported by data.

### `benchmark.py`
`main()` finds readable test images, times the full stored-image inference path, and writes mean/median latency plus approximate FPS.

### `utils.py`
`ensure_project_directories()`, `safe_identity_name()`, `bgr_to_rgb()`, `blur_score()`, and `clamp_box()` keep repeated practical operations readable.

## 26. Complete data flow

```text
Camera
  ↓
OpenCV BGR frame
  ↓
Resize for processing
  ↓
Convert BGR → RGB
  ↓
Detect every face
  ↓
(top, right, bottom, left) box
  ↓
Extract embedding
  ↓
Compare against all local references
  ↓
Find smallest Euclidean distance
  ↓
Apply maximum-distance threshold
  ↓
Known identity / Unknown
  ↓
Scale box back, draw label and distance
  ↓
Display frame and measured FPS
```

**Enrollment changes the reference set:** `face → embedding → stored local reference`. **Recognition does not add data:** `new face → embedding → compare with stored references`. Recognition must never silently enroll a person.

## 27. Privacy and responsible use

Face images and embeddings are biometric data. Use only informed, consenting participants; store enrollment/test data locally; keep raw images and embeddings out of Git; and delete them when no longer needed. This educational project is not designed for surveillance, covert identification, public-database matching, attendance tracking, or security-critical authentication.

## 28. Troubleshooting

### Webcam does not open
Try another `--camera` index, close other camera applications, and check operating-system permissions.

### Face is not detected
Improve lighting, move closer, use a moderate frontal pose, and check that OpenCV receives frames.

### Known person becomes `Unknown`
The threshold may be strict, enrollment images may be poor, or test lighting/pose may differ. Inspect the distance before changing the threshold.

### Unknown person becomes known
The threshold may be too permissive. Collect more unknown validation observations and tune it; never assume the closest name is correct.

### FPS is slow
Try `--scale 0.5` and `--process-every 2`, close heavy applications, and benchmark. Smaller images can hurt far-face accuracy.

### Installation fails
Use Python 3.10 or 3.11 where dlib support is more practical. dlib may require CMake and a C++ compiler.

## 29. Check your understanding

1. What is the difference between face detection and face recognition?
2. What is a face embedding?
3. Why are several enrollment images useful?
4. What does a smaller Euclidean embedding distance usually mean?
5. Why is an `Unknown` decision necessary?
6. What can happen when the threshold is too permissive?
7. What is a false acceptance?
8. What is a false rejection?
9. Why can dim lighting reduce performance?
10. Why can greater camera distance reduce performance?
11. What is FPS?
12. What is inference latency?
13. Why should face images and embeddings stay out of GitHub?
14. What is distribution shift in this context?
15. Why is embedding distance not a probability?

Try answering before opening the section below.

<details><summary>Suggested answers</summary>

1. Detection locates faces; recognition compares a detected face with enrolled identities.
2. A learned numeric vector describing facial appearance collectively.
3. They provide modest variations the query can match.
4. The model considers the vectors more similar.
5. Every query has a closest reference even when the person was never enrolled.
6. More unknown people may be falsely accepted.
7. An unknown person is labeled as enrolled.
8. An enrolled person is labeled `Unknown`.
9. Shadows, noise, and lost contrast can change detection and embeddings.
10. The face occupies fewer pixels and loses detail.
11. Displayed or processed frames per second.
12. Time required for an inference operation.
13. Both are sensitive biometric data.
14. Test conditions differ from the enrollment conditions.
15. Distance is an uncalibrated geometric measurement, not a class probability.

</details>

## 30. Questions I should be able to answer

- Why use pretrained embeddings instead of training a small classifier from scratch?
- How does the `0.60` Euclidean-distance threshold work?
- How are unknown people handled in code, and what evaluation is still missing?
- What changes when the threshold becomes stricter or more permissive?
- How would I design controlled lighting, distance, and pose experiments?
- How would I separate detection failures from threshold rejections?
- What were measured mean latency, median latency, and approximate throughput?
- Why retain multiple enrollment references?
- How would alignment, illumination normalization, calibration, or temporal smoothing improve the study?
- What privacy risks exist for images and embeddings?
- How might a CNN embedding network learn useful representations?
- Why is this not a security-grade biometric system?

Answers about results must come from generated CSV files—not from guesses.

## 31. Final lessons

Face recognition is not only “detect face → print name.” A responsible system requires learned representations, a meaningful distance metric, unknown rejection, threshold trade-offs, runtime measurement, controlled lighting/distance/pose experiments, honest failure analysis, and careful handling of biometric data.

Use this notebook to understand **how** the pipeline works. Use `robustness_analysis.ipynb` to study **how well** it works after collecting real observations.

## 32. Group image person retrieval

Recognition asks **who is this face?** Retrieval asks **where is one requested enrolled identity among all faces?** The workflow is: (1) load one permitted group image, (2) detect every face, (3) retain every bounding box, (4) encode each detectable face, (5) select a label already in the local database, (6) compare that label's references with every face, (7) take the minimum distance, (8) apply the existing threshold, (9) localize only the closest valid match, and (10) report detection, encoding, missing-identity, and threshold-rejection failures separately.

In [ ]:
from group_search import annotate_image
from person_retrieval import retrieve_person

group_files = sorted(p for p in (ROOT / 'examples/group').rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES)
group_frame = None
group_locations = []
if cv2 is None or plt is None:
    print('Install OpenCV and Matplotlib to run this section.')
elif not group_files:
    print('DATA COLLECTION REQUIRED: add a permitted group image under examples/group/.')
else:
    group_frame = cv2.imread(str(group_files[0]))
    group_rgb = cv2.cvtColor(group_frame, cv2.COLOR_BGR2RGB)
    group_locations = detect_faces(group_rgb)
    all_faces_preview = group_frame.copy()
    for top, right, bottom, left in group_locations:
        cv2.rectangle(all_faces_preview, (left, top), (right, bottom), (160, 160, 160), 2)
    print('Group image:', group_files[0])
    print('Faces detected:', len(group_locations))
    plt.figure(figsize=(10, 7)); plt.imshow(cv2.cvtColor(all_faces_preview, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

In [ ]:
if group_frame is None or not group_locations:
    print('DATA COLLECTION REQUIRED: a readable group image with detectable faces is needed.')
else:
    group_embeddings = [get_face_embedding(group_rgb, box) for box in group_locations]
    local_database = load_database(DEFAULT_DATABASE_PATH)
    target_person = sorted(local_database)[0]  # Replace with another enrolled local label if desired.
    retrieval = retrieve_person(
        group_embeddings, group_locations, target_person, local_database, DEFAULT_THRESHOLD
    )
    print(retrieval)
    localized = annotate_image(group_frame, group_locations, retrieval, show_all_faces=True)
    plt.figure(figsize=(10, 7)); plt.imshow(cv2.cvtColor(localized, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f"Target: {target_person} | {'FOUND' if retrieval['found'] else 'NOT FOUND'}"); plt.show()

### Retrieval failure cases and experiments

No detected boxes means detection failed. A `None` embedding means that a detected face could not be encoded. A finite distance above the threshold means the requested identity was not accepted, while a missing label means the person was never enrolled locally. Only a valid below-threshold result is highlighted. For research, use a private `image,target` manifest with `python benchmark.py --group-manifest examples/group/manifest.csv`; the resulting CSV can support group-size-versus-latency analysis only after real measurements exist. Held-out group images, rather than enrollment images, are required for retrieval-accuracy claims.